# Coral Reef Data Collection from Allen Coral Atlas

This notebook collects coral reef data for 100 random reef samples using the Allen Coral Atlas WMS service.

**Data collected for each reef:**
- Polygon outline of the reef
- keyword for reef composition (benthic habitat classes)
- keyword for geomorphic structure

**Data sources:**
- Allen Coral Atlas: https://allencoralatlas.org/
- WMS Service: https://allencoralatlas.org/geoserver/coral-atlas/wms
- GitHub Tutorials: https://github.com/CoralMapping/tutorials

## 1. Install and Import Dependencies

In [16]:
# Install required packages
import subprocess
import sys

packages = ['owslib', 'geopandas', 'shapely', 'pandas', 'numpy', 'requests']
for package in packages:
    try:
        __import__(package)
        print(f"{package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

owslib already installed
geopandas already installed
shapely already installed
pandas already installed
numpy already installed
requests already installed


In [17]:
# Import libraries
import json
import random
import pandas as pd
import geopandas as gpd
import numpy as np
from owslib.wms import WebMapService
from shapely.geometry import box, shape
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Setup WMS Service Connection

In [24]:
# Initialize WMS service with connection pooling
WMS_URL = 'https://allencoralatlas.org/geoserver/coral-atlas/wms'
WMS_VERSION = '1.3.0'

# Create WMS object with custom User-Agent header and session
headers = {'User-Agent': 'owslib-CoralAtlas'}
wms = WebMapService(WMS_URL, version=WMS_VERSION, headers=headers)

# Store service for reuse
print(f"Connected to WMS service: {WMS_URL}")
print(f"\nAvailable layers:")
for layer_name in list(wms.contents.keys())[:10]:
    print(f"  - {layer_name}")

# Import time for delays and backoff
import time

Connected to WMS service: https://allencoralatlas.org/geoserver/coral-atlas/wms

Available layers:
  - benthic_data_verbose
  - geomorphic_data_verbose


## 3. Define Helper Functions

In [25]:
def query_wms_for_habitat_with_retry(wms, bbox, layer_name, max_retries=3, initial_delay=1):
    """
    Query WMS service for habitat data with retry logic and exponential backoff.
    
    Args:
        wms: WebMapService object
        bbox: Bounding box (minx, miny, maxx, maxy)
        layer_name: Name of the WMS layer to query
        max_retries: Maximum number of retry attempts
        initial_delay: Initial delay in seconds between retries
    
    Returns:
        GeoDataFrame with habitat polygons or None if query fails
    """
    for attempt in range(max_retries):
        try:
            response = wms.getmap(
                layers=[layer_name],
                srs='EPSG:4326',
                bbox=bbox,
                size=(256, 256),
                timeout=30,  # Reduced timeout to 30 seconds
                format='application/json;type=geojson'
            )
            
            data = json.load(response)
            
            if data['features']:
                gdf = gpd.GeoDataFrame.from_features(data['features'])
                gdf.set_crs('EPSG:4326', inplace=True)
                return gdf
            else:
                return None
        
        except Exception as e:
            if attempt < max_retries - 1:
                # Calculate exponential backoff delay
                delay = initial_delay * (2 ** attempt)
                print(f"  ⚠ Retry {attempt + 1}/{max_retries - 1}: Waiting {delay}s before retry...")
                time.sleep(delay)
            else:
                # Final attempt failed, return None silently
                return None
    
    return None


def extract_reef_data(benthic_gdf, geomorphic_gdf, bbox):
    """
    Extract one benthic and one geomorphic polygon from each reef location.
    
    Args:
        benthic_gdf: GeoDataFrame with benthic data
        geomorphic_gdf: GeoDataFrame with geomorphic data
        bbox: Bounding box of the reef area
    
    Returns:
        dict: Combined reef data with single polygons and class names, or None if insufficient data
    """
    benthic_polygon = None
    benthic_class = None
    geomorphic_polygon = None
    geomorphic_class = None
    
    # Select one random benthic polygon if available
    if benthic_gdf is not None and len(benthic_gdf) > 0:
        benthic_idx = random.randint(0, len(benthic_gdf) - 1)
        benthic_row = benthic_gdf.iloc[benthic_idx]
        benthic_polygon = benthic_row.geometry
        # Try to get class name from various possible column names
        for col in ['class_name', 'CLASSIFICATION', 'classification', 'CLASS', 'class', 'habitat']:
            if col in benthic_row.index and pd.notna(benthic_row[col]):
                benthic_class = str(benthic_row[col])
                break
    
    # Select one random geomorphic polygon if available
    if geomorphic_gdf is not None and len(geomorphic_gdf) > 0:
        geomorphic_idx = random.randint(0, len(geomorphic_gdf) - 1)
        geomorphic_row = geomorphic_gdf.iloc[geomorphic_idx]
        geomorphic_polygon = geomorphic_row.geometry
        # Try to get class name from various possible column names
        for col in ['class_name', 'CLASSIFICATION', 'classification', 'CLASS', 'class', 'habitat']:
            if col in geomorphic_row.index and pd.notna(geomorphic_row[col]):
                geomorphic_class = str(geomorphic_row[col])
                break
    
    # Return only if we have at least one polygon from each type
    if benthic_polygon is not None and geomorphic_polygon is not None:
        return {
            'benthic_polygon': benthic_polygon,
            'benthic_class': benthic_class,
            'geomorphic_polygon': geomorphic_polygon,
            'geomorphic_class': geomorphic_class,
        }
    
    return None

print("Helper functions with retry logic defined!")

Helper functions with retry logic defined!


## 4. Collect Reef Data

In [27]:
# Configuration
NUM_SAMPLES = 20
BENTHIC_LAYER = 'benthic_data_verbose'
GEOMORPHIC_LAYER = 'geomorphic_data_verbose'
DELAY_BETWEEN_REQUESTS = 0.5  # Delay in seconds between queries to respect rate limits
MAX_ATTEMPTS = NUM_SAMPLES * 3  # Try up to 3x as many to ensure we get enough samples

# List to store combined reef data
combined_reef_data = []
successful_samples = 0
failed_queries = 0

print(f"Starting collection of {NUM_SAMPLES} reef samples...\n")
print(f"Rate limit: {DELAY_BETWEEN_REQUESTS}s between requests")
print(f"Max query attempts: {MAX_ATTEMPTS}\n")

start_time = time.time()

for i in range(MAX_ATTEMPTS):
    # Generate random bounding box
    bbox = generate_random_bbox(padding=2)
    
    # Add delay to respect rate limits (except on first query)
    if i > 0:
        time.sleep(DELAY_BETWEEN_REQUESTS)
    
    # Query benthic data with retry logic
    benthic_gdf = query_wms_for_habitat_with_retry(wms, bbox, BENTHIC_LAYER, max_retries=2)
    
    # Query geomorphic data with retry logic
    geomorphic_gdf = query_wms_for_habitat_with_retry(wms, bbox, GEOMORPHIC_LAYER, max_retries=2)
    
    # Extract single polygons from this location
    reef_data = extract_reef_data(benthic_gdf, geomorphic_gdf, bbox)
    
    if reef_data is not None:
        reef_data['reef_id'] = successful_samples + 1
        combined_reef_data.append(reef_data)
        successful_samples += 1
        
        if successful_samples % 10 == 0:
            elapsed = time.time() - start_time
            rate = successful_samples / elapsed
            print(f"✓ Collected {successful_samples} reef samples ({rate:.1f} samples/min)")
        
        # Stop when we have enough samples
        if successful_samples >= NUM_SAMPLES:
            break
    else:
        failed_queries += 1
    
    # Progress indicator every 20 queries
    if (i + 1) % 20 == 0 and successful_samples < NUM_SAMPLES:
        print(f"  ~ Processed {i + 1} queries (collected {successful_samples}, failed {failed_queries})")

elapsed_time = time.time() - start_time
print(f"\n✓ Data collection complete!")
print(f"✓ Successfully collected: {successful_samples} reef locations")
print(f"✓ Failed/empty queries: {failed_queries}")
print(f"✓ Time elapsed: {elapsed_time:.1f} seconds")
print(f"✓ Total combined reef samples: {len(combined_reef_data)}")

Starting collection of 20 reef samples...

Rate limit: 0.5s between requests
Max query attempts: 60

  ⚠ Retry 1/1: Waiting 1s before retry...
  ~ Processed 20 queries (collected 6, failed 14)
✓ Collected 10 reef samples (0.1 samples/min)
  ~ Processed 40 queries (collected 11, failed 29)
  ⚠ Retry 1/1: Waiting 1s before retry...
  ⚠ Retry 1/1: Waiting 1s before retry...
  ⚠ Retry 1/1: Waiting 1s before retry...
  ~ Processed 60 queries (collected 18, failed 42)

✓ Data collection complete!
✓ Successfully collected: 18 reef locations
✓ Failed/empty queries: 42
✓ Time elapsed: 451.8 seconds
✓ Total combined reef samples: 18


## 5. Export Combined Reef Data to GeoJSON

In [28]:
# Create combined GeoDataFrame with MultiGeometry approach
features = []

for reef in combined_reef_data:
    feature = {
        'type': 'Feature',
        'properties': {
            'reef_id': reef['reef_id'],
            'benthic_class': reef['benthic_class'],
            'geomorphic_class': reef['geomorphic_class'],
        },
        'geometry': {
            'type': 'GeometryCollection',
            'geometries': [
                reef['benthic_polygon'].__geo_interface__,
                reef['geomorphic_polygon'].__geo_interface__,
            ]
        }
    }
    features.append(feature)

# Create GeoJSON FeatureCollection
geojson_data = {
    'type': 'FeatureCollection',
    'features': features
}

# Export to GeoJSON
output_path = 'scrapped_data/coral_reef_combined.geojson'
with open(output_path, 'w') as f:
    json.dump(geojson_data, f, indent=2)

print(f"✓ Combined reef data exported to: {output_path}")
print(f"✓ Total features: {len(features)}")
print(f"\\nFirst 3 samples:\")")
for i, reef in enumerate(combined_reef_data[:3]):
    print(f"  Sample {reef['reef_id']}:")
    print(f"    Benthic class: {reef['benthic_class']}")
    print(f"    Geomorphic class: {reef['geomorphic_class']}")

✓ Combined reef data exported to: scrapped_data/coral_reef_combined.geojson
✓ Total features: 18
\nFirst 3 samples:")
  Sample 1:
    Benthic class: Coral/Algae
    Geomorphic class: Inner Reef Flat
  Sample 2:
    Benthic class: Coral/Algae
    Geomorphic class: Back Reef Slope
  Sample 3:
    Benthic class: Coral/Algae
    Geomorphic class: Reef Slope
